# Paper figures

Render the supplied numerical results. Training and evaluation are separate scripts.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'artifacts/tables').is_dir())
TABLES=ROOT/'artifacts/tables'; FIGURES=ROOT/'outputs/paper/figures';FIGURES.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':8,'axes.spines.top':True,'axes.spines.right':True})
def read(name):return pd.read_csv(TABLES/name,dtype={'cohort_label':str})
def save(fig,name):
 for ext in ['pdf','png']:fig.savefig(FIGURES/(name+'.'+ext),bbox_inches='tight',dpi=300)
 plt.show()


## Frozen mortality: SSL and foundation models

In [ ]:
d=read('fig16_ter_bis_per_seed.csv')
slots=list(dict.fromkeys(zip(d.family,d.variant)))
colors={'biomecg':'#547FB8','pclr':'#7FA568','jepa_resnet':'#9773AC','ecgcpc_local':'#CA9D28','ecgfounder':'#AD6C4D','ecgcpc':'#3A8183','ecgjepa':'#876B80'}
fig,axes=plt.subplots(3,2,figsize=(6.5,6.2),sharey=True)
for row,c in enumerate(['1000','8000','full']):
 for col,site in enumerate(['mimic','code15']):
  ax=axes[row,col]
  for x,(f,v) in enumerate(slots):
   q=d[(d.site==site)&(d.cohort_label==c)&(d.family==f)&(d.variant==v)].test_patient_auc
   if len(q):ax.bar(x,q.mean(),yerr=q.std() if len(q)>1 else None,color=colors[f],hatch={'early':'','early8':'///','late':'...'}.get(v,''),capsize=2)
  ax.set_title(f'{site.upper()} | {c}');ax.set_ylim(.65,.9);ax.grid(axis='y',alpha=.2)
  ax.set_xticks(range(len(slots)),[f+' '+v for f,v in slots],rotation=75,fontsize=5)
fig.tight_layout();save(fig,'fig16_ter_bis_mortality_frozen_fusion_transposed')


## Pretraining scaling

In [ ]:
d=read('mortality_scaling_frozen_strong_references_curve.csv');r=read('mortality_scaling_frozen_strong_references_baselines.csv')
fig,axes=plt.subplots(1,2,figsize=(6,2.65),sharey=True)
for ax,site in zip(axes,['mimic','code15']):
 q=d[d.site==site].sort_values('pretraining_size');x=q.pretraining_size.to_numpy()
 ax.plot(x,q['mean'],marker='o',color='#547FB8');ax.fill_between(x,q['mean']-q.sd,q['mean']+q.sd,alpha=.15)
 for z in r[r.site==site].itertuples():ax.axhline(z.mean,linestyle='--',label=z.family)
 ax.set_xscale('log');ax.set_title('Downstream: '+site.upper()+' Full');ax.grid(axis='y',alpha=.2)
axes[0].set_ylabel('Test AUROC',fontweight='bold');axes[1].legend(fontsize=6)
fig.supxlabel('MIMIC pretraining cohort size');fig.tight_layout();save(fig,'fig17_mortality_scaling_frozen_strong_references')


## Censoring sensitivity 26

In [ ]:
d=read('fig26_test_only_followup_per_seed.csv')
slots=list(dict.fromkeys(zip(d.family,d.variant)));conditions=list(d.condition.unique())
fig,axes=plt.subplots(2,2,figsize=(6.5,5.4),sharey=True)
for row,site in enumerate(['mimic','code15']):
 for col,c in enumerate(['8000','full']):
  ax=axes[row,col]
  for x,(f,v) in enumerate(slots):
   for k,condition in enumerate(conditions):
    q=d[(d.site==site)&(d.cohort_label==c)&(d.family==f)&(d.variant==v)&(d.condition==condition)].test_patient_auc
    if len(q):ax.bar(x+(k-.5)*.38,q.mean(),width=.35,color=colors[f],hatch='///' if k else '')
  ax.set_title(site.upper()+' | '+c);ax.set_ylim(.5,.95);ax.set_xticks(range(len(slots)),[f+' '+v for f,v in slots],rotation=70,fontsize=5)
fig.tight_layout();save(fig,'fig26_censoring')


## Censoring sensitivity 26_bis

In [ ]:
d=read('fig26_bis_followup_per_seed.csv')
slots=list(dict.fromkeys(zip(d.family,d.variant)));conditions=list(d.condition.unique())
fig,axes=plt.subplots(2,2,figsize=(6.5,5.4),sharey=True)
for row,site in enumerate(['mimic','code15']):
 for col,c in enumerate(['8000','full']):
  ax=axes[row,col]
  for x,(f,v) in enumerate(slots):
   for k,condition in enumerate(conditions):
    q=d[(d.site==site)&(d.cohort_label==c)&(d.family==f)&(d.variant==v)&(d.condition==condition)].test_patient_auc
    if len(q):ax.bar(x+(k-.5)*.38,q.mean(),width=.35,color=colors[f],hatch='///' if k else '')
  ax.set_title(site.upper()+' | '+c);ax.set_ylim(.5,.95);ax.set_xticks(range(len(slots)),[f+' '+v for f,v in slots],rotation=70,fontsize=5)
fig.tight_layout();save(fig,'fig26_bis_censoring')


## Shared plotting helpers

In [ ]:
def curve(ax,q,xcol='cohort_label',value='value',label=None,color=None):
 q=q.copy();q['x']=pd.to_numeric(q[xcol].replace('full',np.nan),errors='coerce')
 finite=q.x.dropna();full=(finite.max()*2 if len(finite) else 100000)
 q['x']=q.x.fillna(full)
 s=q.groupby('x')[value].agg(['mean','std']).sort_index()
 if s.empty:return
 ax.plot(s.index,s['mean'],marker='o',markersize=2.5,linewidth=1.2,label=label,color=color)
 ax.fill_between(s.index,(s['mean']-s['std']).to_numpy(),(s['mean']+s['std']).to_numpy(),alpha=.12,color=color)
 ax.set_xscale('log');ax.grid(axis='y',alpha=.2)
 for spine in ax.spines.values():spine.set_visible(True);spine.set_linewidth(.8)
 ax.tick_params(labelsize=7)


## Age and sex prediction

In [ ]:
d=read('demographics_complete_per_seed.csv')
fig,axes=plt.subplots(1,4,figsize=(6.5,2.35))
for ax,(task,site) in zip(axes,[('age','mimic'),('age','code15'),('sex','mimic'),('sex','code15')]):
 for arch,fusion,label,color in [(f'pretrained_{site}_frozen','Late','biomECG','#547FB8'),(f'pretrained_{site}_frozen','Early','biomECG Early','#7FA568'),('ecgfounder_frozen','Late','ECGFounder','#9773AC')]:
  q=d[(d.site==site)&(d.task==task)&(d.architecture==arch)&(d.fusion==fusion)]
  curve(ax,q,label=label,color=color)
 ax.set_title(site.upper()+' | '+task.capitalize(),fontsize=8);ax.set_ylabel('MAE (years)' if task=='age' else 'Test AUROC',fontweight='bold')
handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,loc='lower center',ncol=3,fontsize=7)
fig.tight_layout(rect=(0,.13,1,1));save(fig,'fig10_demographics_mimic_code15')


## Frozen fusion and late-fusion adaptation

In [ ]:
d=read('mortality_biomecg_fusion_paper_per_seed.csv')
configs=[('pretrained_mimic_frozen','Early','Early','#7FA568'),('pretrained_mimic_frozen','Early ×8','Early ensemble ×8','#9773AC'),('pretrained_mimic_frozen','Late','Late','#547FB8')]
for name,config in [('fig22_mortality_mimic_frozen_fusion',configs),('fig23_mortality_mimic_late_regimes',[('pretrained_mimic_frozen','Late','Frozen','#547FB8'),('pretrained_mimic_finetuned','Late','Fine-tuned','#C87535'),('scratch','Late','Scratch','#4F9993')])]:
 fig,ax=plt.subplots(figsize=(3,2.65))
 for arch,fusion,label,color in config:
  curve(ax,d[(d.architecture==arch)&(d.fusion==fusion)],value='test_patient_auc',label=label,color=color)
 ax.set_title('MIMIC mortality',fontsize=8);ax.set_ylabel('Test AUROC',fontweight='bold');ax.set_xlabel('Downstream training cohort size',fontsize=7);ax.legend(fontsize=6)
 fig.tight_layout();save(fig,name)


## ECG versus demographic covariates

In [ ]:
d=read('full_late_covariate_ablation_per_seed.csv');d=d[d.seed==12345]
b=read('mortality_covariate_ablation_per_seed.csv');b=b[(b.seed==12345)&(b.variant=='age_sex_only')&(b.cohort_label=='full')]
fig,axes=plt.subplots(1,2,figsize=(6.5,2.8),sharey=True)
families=['biomECG','PCLR','JEPA','ECG-CPC']
for ax,site in zip(axes,['mimic','code15']):
 for x,f in enumerate(families):
  for v,width,hatch in [('ECG + age + sex',.76,''),('ECG only',.38,'///')]:
   q=d[(d.site==site)&(d.family==f)&(d.variant==v)]
   if len(q):ax.bar(x,q.auc.mean(),width=width,color='#547FB8' if not hatch else 'white',hatch=hatch,edgecolor='#333333',linewidth=.7)
 ax.axhline(b[b.site==site].test_patient_auc.mean(),color='#222222',linestyle='--',label='Age + sex only')
 ax.set_xticks(range(4),families,fontsize=7);ax.set_title(site.upper()+' | Full',fontsize=8);ax.set_ylim(.65,.9);ax.grid(axis='y',alpha=.2)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor='#547FB8',label='ECG + age + sex'),Patch(facecolor='white',hatch='///',edgecolor='black',label='ECG only'),plt.Line2D([],[],color='black',linestyle='--',label='Age + sex only')],loc='lower center',ncol=3,fontsize=7)
axes[0].set_ylabel('Test AUROC',fontweight='bold');fig.tight_layout(rect=(0,.12,1,1));save(fig,'fig27_full_late_covariate_ablation')


## Age prediction bias

In [ ]:
d=read('age_late_fusion_bias_per_seed.csv')
fig,axes=plt.subplots(1,2,figsize=(6,2.8),sharey=True)
for ax,site in zip(axes,['mimic','code15']):
 q=d[d.site==site]
 for arch,z in q.groupby('architecture'):curve(ax,z,value='bias',label=arch.replace('pretrained_'+site,'biomECG').replace('ecgfounder','ECGFounder').replace('_',' '))
 ax.axhline(0,color='black',linestyle=':',linewidth=.8);ax.set_title(site.upper(),fontsize=8);ax.set_xlabel('Downstream training size',fontsize=7)
axes[0].set_ylabel('Predicted minus true age (years)',fontweight='bold');axes[1].legend(fontsize=5)
fig.tight_layout();save(fig,'fig14_age_late_fusion_bias')


## Biometric representation UMAP

In [ ]:
labels=pd.read_csv(TABLES/'umap_display_labels.csv')
with np.load(TABLES/'umap_coordinates.npz',allow_pickle=False) as a:coordinates=[a[k] for k in sorted(a.files)]
fig,axes=plt.subplots(1,2,figsize=(6.5,3))
for ax,z,title in zip(axes,coordinates,['Euclidean | 15 neighbors','Cosine | 50 neighbors']):
 if len(z)!=len(labels):raise ValueError('Coordinate/display-label mismatch')
 for k,patient in enumerate(sorted(labels.patient_display.unique())):
  mask=labels.patient_display.eq(patient).to_numpy();ax.scatter(z[mask,0],z[mask,1],s=12,color=plt.cm.tab20(k%20),marker=['o','s'][k//20%2])
 ax.set_title(title,fontsize=8);ax.set_xticks([]);ax.set_yticks([])
fig.tight_layout();save(fig,'fig13_umap_biometric_patient_identity')
